# Generated structure validity and uniqueness

Analyze generated structures using SMACT validity, minimum interatomic distance, and compositional and structural uniqueness.
Set the data and output directories before running the analysis.


In [ ]:
import os
from pymatgen.core import Structure
import smact
import numpy as np
import itertools
from smact.screening import pauling_test
import warnings
warnings.filterwarnings("ignore")
from pymatgen.analysis.structure_matcher import StructureMatcher
from collections import defaultdict
from scipy.spatial.distance import cdist
import pandas as pd
from matminer.featurizers.site.fingerprint import CrystalNNFingerprint
from matminer.featurizers.composition.composite import ElementProperty
from matminer.featurizers.structure import RadialDistributionFunction
from joblib import Parallel, delayed

In [ ]:
def smact_validity(comp, count,
                   use_pauling_test=True,
                   include_alloys=True):
    elem_symbols = tuple(comp)
    space = smact.element_dictionary(elem_symbols)
    smact_elems = [e[1] for e in space.items()]
    electronegs = [e.pauling_eneg for e in smact_elems]
    ox_combos = [e.oxidation_states for e in smact_elems]
    if len(set(elem_symbols)) == 1:
        return True
    if include_alloys:
        is_metal_list = [elem_s in smact.metals for elem_s in elem_symbols]
        if all(is_metal_list):
            return True

    threshold = np.max(count)
    compositions = []
    for ox_states in itertools.product(*ox_combos):
        stoichs = [(c,) for c in count]
        # Test for charge balance
        cn_e, cn_r = smact.neutral_ratios(
            ox_states, stoichs=stoichs, threshold=threshold)
        # Electronegativity test
        if cn_e:
            if use_pauling_test:
                try:
                    electroneg_OK = pauling_test(ox_states, electronegs)
                except TypeError:
                    # if no electronegativity data, assume it is okay
                    electroneg_OK = True
            else:
                electroneg_OK = True
            if electroneg_OK:
                for ratio in cn_r:
                    compositions.append(
                        tuple([elem_symbols, ox_states, ratio]))
    compositions = [(i[0], i[2]) for i in compositions]
    compositions = list(set(compositions))
    if len(compositions) > 0:
        return True
    else:
        return False

def structure_validity(crystal, cutoff=0.5):
    dist_mat = crystal.distance_matrix
    # Pad diagonal with a large number
    dist_mat = dist_mat + np.diag(
        np.ones(dist_mat.shape[0]) * (cutoff + 10.))
    if dist_mat.min() < cutoff or crystal.volume < 0.1:
        return False
    else:
        return True

In [ ]:
# data_dir = "D:/Works/doing/gm_cifs/cdvae/v32/to_plot/sps_30_gens"

oxyf_folders = ["oxyf_s_20_gen", "oxyf_ps_20_gen", "oxyf_asse_20_gen", "oxyf_passb_20_gen", "oxyf_sbte_20_gen", "oxyf_ssete_20_gen"]

for oxyf_folder in oxyf_folders:
    data_dir = f"D:/PycharmProjects/cdvae-local/pt_files/v32/{oxyf_folder}/cifs"

    total = 0
    smact_valid_count = 0
    structure_valid_count = 0
    both_valid_count = 0

    for file in os.listdir(data_dir):
        if file.endswith(".cif"):
            total += 1

            data_file = os.path.join(data_dir, file)
            structure = Structure.from_file(data_file)

            redu_comp = structure.composition.reduced_composition
            elems = list(redu_comp.as_dict().keys())
            counts = [int(x) for x in redu_comp.as_dict().values()]

            smact_valid = smact_validity(elems, counts)
            structure_valid = structure_validity(structure)

            if smact_valid:
                smact_valid_count += 1
            if structure_valid:
                structure_valid_count += 1
            if smact_valid and structure_valid:
                both_valid_count += 1

    smact_rate = smact_valid_count / total if total > 0 else 0
    structure_rate = structure_valid_count / total if total > 0 else 0
    both_rate = both_valid_count / total if total > 0 else 0

    print(oxyf_folder)
    print(f"Total: {total}")
    print(f"SMACT-valid structures: {smact_valid_count}, validity rate: {smact_rate:.4f}")
    print(f"Structurally valid structures: {structure_valid_count}, validity rate: {structure_rate:.4f}")
    print(f"Valid by both criteria: {both_valid_count}, validity rate: {both_rate:.4f}")
    print()

In [ ]:
all_folders = [
    "oxy_bcn_40_gen", "oxy_bp_40_gen", "oxy_csi_40_gen", "oxy_np_40_gen", "oxy_sps_30_gen",
    "oxyf_s_20_gen", "oxyf_ps_20_gen", "oxyf_asse_20_gen", "oxyf_passb_20_gen", "oxyf_sbte_20_gen", "oxyf_ssete_20_gen"
]

for folder in all_folders:
    data_dir = f"D:/PycharmProjects/cdvae-local/pt_files/v32/{folder}/cifs"

    comp_groups = defaultdict(list)
    total = 0

    for file in os.listdir(data_dir):
        if file.endswith(".cif"):
            total += 1
            path = os.path.join(data_dir, file)

            try:
                structure = Structure.from_file(path)
            except Exception as e:
                print(f"Failed to read: {file}")
                continue

            comp = structure.composition.reduced_formula
            comp_groups[comp].append(structure)

    matcher = StructureMatcher()

    unique_structure_count = 0
    unique_comp_count = len(comp_groups)

    for comp, group in comp_groups.items():
        unique_group = []

        for s in group:
            if not any(matcher.fit(s, t) for t in unique_group):
                unique_group.append(s)

        unique_structure_count += len(unique_group)

    print(folder)
    print(f"Total CIF files: {total}")
    print(f"Unique compositions: {unique_comp_count} {unique_comp_count / total :.4f}")
    print(f"Unique structures: {unique_structure_count} {unique_structure_count / total :.4f}")
    print()